In [ ]:
import pandas as pd
from tqdm import tqdm
import json,re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from PIL import Image
import math

def read_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            try:
                json_object = json.loads(line.strip())
                data.append(json_object)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
                continue
                # 如果选择抛出异常，使用下面这行
                # raise
    return data

def convert_action(action, img_width=720, img_height=1280):
    if action.startswith('Click'):
        # Extract coordinates from 'Click (x,y)' using regex
        coords = re.findall(r'\(([^)]+)', action)
        if coords:
            x, y = coords[0].split(', ')
            json_answer = {
                "action": "click",
                "coordinate": [int(float(x)*img_width),int(float(y)*img_height)]
            }
            return json_answer
            # return f"click(start_box='<|box_start|>({x},{y})<|box_end|>')"
    
    elif action == 'KEY_BACK':
        json_answer = {
            "action": "system_button",
            "button": "Back"
        }
        # return "press_back()"
        return json_answer
    
    elif action == 'KEY_HOME':
        json_answer = {
            "action": "system_button",
            "button": "Home"
        }
        return json_answer
        
    
    elif action.startswith('Stop'):
        json_answer = {
            "action": "terminate",
            "status": "success"
        }
        return json_answer

    elif action.startswith('Type'):
        # Extract text from 'Type: content'
        content = action.split(': ', 1)[-1]
        json_answer = {
            "action": "type",
            "text": content
        }
        return json_answer
        # return f"type(content='{content}')"
    
    elif action.startswith('Swipe'):
        # Extract start and end coordinates from 'Swipe (x1,y1), (x2,y2)'
        coords = re.findall(r'\(([^)]+)', action)
        if len(coords) == 2:
            x1, y1 = coords[0].split(', ')
            x2, y2 = coords[1].split(', ')
            json_answer = {
                "action": "swipe",
                "coordinate1": [int(float(x1)*img_width),int(float(y1)*img_height)],
                "coordinate2": [int(float(x2)*img_width),int(float(y2)*img_height)]
            }
            return json_answer
            # return f"scroll(start_box='<|box_start|>({x1},{y1})<|box_end|>', end_box='<|box_start|>({x2},{y2})<|box_end|>')"
    
    return "Action not recognized"

def is_gt_data(data_list):
    return all(
        data['is_correct'] == 'Y' or (data['is_correct'] == 'N' and data['human_action'])
        for data in data_list
    )

def is_trajectory_data(data_list):
    if data_list[-1]['is_correct'] == 'Y' and data_list[-1]['action'] == 'Stop':
        return True
    if data_list[-1]['is_correct'] == 'N' and data_list[-1]['human_action'] == 'Stop':
        return True
    return False

def extract_between_answer(text):
    start = text.find("<answer>") + len("<answer>")
    end = text.find("</answer>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def extract_between_think(text):
    start = text.find("<think>") + len("<think>")
    end = text.find("</think>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""


def extract_between_tool_call(text):
    start = text.find("<tool_call>") + len("<tool_call>")
    end = text.find("</tool_call>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def resize_to_multiple_of_28(width, height, max_pixels):
    # 计算当前像素总数
    current_pixels = width * height
    
    if current_pixels <= max_pixels:
        return width, height

    # 保持长宽比缩放
    aspect_ratio = width / height

    # 假设新的高度为 h，则宽度为 round(h * aspect_ratio)
    # 我们要找到最大的 h 使得 (h * w) <= max_pixels，并且 h, w 是 28 的倍数

    new_height = int((max_pixels / aspect_ratio) ** 0.5)
    new_height = (new_height // 28) * 28  # 向下取整到 28 的倍数

    new_width = int(aspect_ratio * new_height)
    new_width = (new_width // 28) * 28  # 再次确保是 28 的倍数

    # 可能由于四舍五入导致 new_width * new_height > max_pixels，再检查一次
    while new_width * new_height > max_pixels and new_height >= 28 and new_width >= 28:
        new_height -= 28
        new_width = int(aspect_ratio * new_height)
        new_width = (new_width // 28) * 28

    return new_width, new_height


def split_tool_call(text: str) -> tuple[str, str, str]:
    """
    将文本拆分成三段：
    before  : <tool_call> 之前的字符串
    middle  : <tool_call> 与 </tool_call> 之间的字符串（去掉首尾空白）
    after   : </tool_call> 之后的字符串

    如果标记不存在，则 middle 为空串，before 为原始文本，after 为空串。
    """
    open_tag, close_tag = "<tool_call>", "</tool_call>"

    start = text.find(open_tag)
    end   = text.find(close_tag, start + len(open_tag))  # 从 open_tag 之后再找，避免嵌套误匹配

    # 没找到成对标记，直接返回
    if start == -1 or end == -1:
        return text, "", ""

    before = text[:start]
    middle = text[start + len(open_tag) : end].strip()
    after  = text[end + len(close_tag) :]

    return before, middle, after
# available_gt_data_5_10_steps = [data for data in available_data_5_10_steps if is_gt_data(data['data'])]
# 读取xlsx文件
df = pd.read_excel('/home/aiqihang.aqh/GUI-R1/data/gui_test.xlsx')
column_list = df['data'].tolist()
test_instructions = [json.loads(data)['goal'].strip() for data in column_list]

image_folder = "/home/aiqihang.aqh/GUI-R1/new_images"

In [4]:
sft_data = read_jsonl("/home/aiqihang.aqh/Appagent/data/interactive_sft.jsonl")

with open("/home/aiqihang.aqh/LLaMA-Factory/data/interactive_sft.json", "w", encoding="utf-8") as file:
    json.dump(sft_data, file, ensure_ascii=False, indent=2)  # indent=2 使输出更易读